In [1]:
import os
import json
import random
import re
import unicodedata
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType

# ==========================================
# CONFIGURATION
# ==========================================
NEW_RESEARCH_DIR = r"D:\student1402\negar\final_research"

# ← دیتاست جدید pipeline-aware
TRAIN_DATA_PATH = NEW_RESEARCH_DIR + r"\data\pipeline_aware_re_train_abstract_clean.json"
# ← همان TSV اصلی برای خواندن abstract کامل (مثل phase2_abstract_lora_dr)
#ABSTRACT_TSV    = NEW_RESEARCH_DIR + r"\data\lotus_with_title_abstract.tsv"

OUT_DIR    = NEW_RESEARCH_DIR + r"\outputs_phase2_pipeline_aware_abstract_clean"
MODEL_DIR  = NEW_RESEARCH_DIR + r"\phase2_model_pipeline_aware_abstrcat_clean"
MODEL_NAME = r"C:\Users\UMZ\.cache\huggingface\hub\models--microsoft--BiomedNLP-PubMedBERT-base-uncased-abstract\snapshots\d673b8835373c6fa116d6d8006b33d48734e305d"

os.makedirs(OUT_DIR,   exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

MAX_LEN      = 512
EPOCHS       = 3
BATCH        = 8
GRAD_ACC     = 2
LR           = 2e-4
LORA_R       = 8
LORA_ALPHA   = 32
LORA_DROPOUT = 0.1
SEED         = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("CUDA available:", torch.cuda.is_available())

c:\Users\UMZ\anaconda3\envs\pubmedbert\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True


In [2]:
# ==========================================
# HELPER — inject_markers (همان کد قبلی)
# ==========================================
def inject_markers_closest(abstract, org_str, chem_str):
    """Finds ALL mentions and marks the CLOSEST non-overlapping pair."""
    org_matches  = [m.span() for m in re.finditer(re.escape(org_str),  abstract)]
    chem_matches = [m.span() for m in re.finditer(re.escape(chem_str), abstract)]

    if not org_matches:
        org_matches  = [m.span() for m in re.finditer(re.escape(org_str.lower()),  abstract.lower())]
    if not chem_matches:
        chem_matches = [m.span() for m in re.finditer(re.escape(chem_str.lower()), abstract.lower())]

    if not org_matches or not chem_matches:
        return None

    valid_pairs = []
    for o_start, o_end in org_matches:
        for c_start, c_end in chem_matches:
            if max(o_start, c_start) < min(o_end, c_end): continue
            valid_pairs.append((abs(o_start - c_start), o_start, o_end, c_start, c_end))

    if not valid_pairs: return None
    valid_pairs.sort()
    _, o_start, o_end, c_start, c_end = valid_pairs[0]

    s = abstract
    if o_start > c_start:
        s = s[:o_start] + "[O]" + abstract[o_start:o_end] + "[/O]" + s[o_end:]
        s = s[:c_start] + "[C]" + abstract[c_start:c_end] + "[/C]" + s[c_end:]
    else:
        s = s[:c_start] + "[C]" + abstract[c_start:c_end] + "[/C]" + s[c_end:]
        s = s[:o_start] + "[O]" + abstract[o_start:o_end] + "[/O]" + s[o_end:]
    return s

In [3]:
# ==========================================
# LOAD DATA & BUILD EXAMPLES
# ==========================================
with open(TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data):,} records from pipeline-aware dataset.")
print(f"  label=1 : {sum(1 for r in raw_data if r.get('label')==1):,}")
print(f"  label=0 : {sum(1 for r in raw_data if r.get('label')==0):,}")

# Build samples with markers on full abstract
examples = []
skipped = 0

for rec in tqdm(raw_data, desc="Injecting markers into abstracts"):
    abstract = rec.get("abstract") or rec.get("supporting_sentence", "")
    org  = rec.get("organism_span", "")
    chem = rec.get("chemical_span", "")

    if not abstract or not org or not chem:
        skipped += 1
        continue

    marked = inject_markers_closest(abstract, org, chem)
    if marked is None:
        skipped += 1
        continue

    examples.append({
        "pmid" : str(rec.get("pubmed_id", "")),
        "text" : marked,
        "label": rec["label"],
    })

print(f"✅ Final examples: {len(examples):,}")
print(f"   Skipped: {skipped}")


Loaded 514,370 records from pipeline-aware dataset.
  label=1 : 102,874
  label=0 : 411,496


Injecting markers into abstracts: 100%|██████████| 514370/514370 [00:31<00:00, 16429.75it/s]

✅ Final examples: 471,923
   Skipped: 42447


In [4]:
# ==========================================
# DOCUMENT-LEVEL SPLIT (no data leakage)
# ==========================================
full_df = pd.DataFrame(examples)

unique_pmids = full_df["pmid"].unique()
train_pmids, val_pmids = train_test_split(
    unique_pmids, test_size=0.10, random_state=SEED
)

train_df = full_df[full_df["pmid"].isin(train_pmids)].sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df   = full_df[full_df["pmid"].isin(val_pmids)].sample(frac=1,   random_state=SEED).reset_index(drop=True)

print(f"Train : {len(train_df):,}  |  Val : {len(val_df):,}")
print(f"Train pos/neg: {(train_df['label']==1).sum():,} / {(train_df['label']==0).sum():,}")
print(f"Val   pos/neg: {(val_df['label']==1).sum():,}   / {(val_df['label']==0).sum():,}")

dataset = DatasetDict({
    "train"     : Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
})

Train : 423,075  |  Val : 48,848
Train pos/neg: 74,289 / 348,786
Val   pos/neg: 8,184   / 40,664


In [5]:
# ==========================================
# TOKENIZE
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.add_special_tokens({"additional_special_tokens": ["[O]", "[/O]", "[C]", "[/C]"]})

def tokenize(batch):
    enc = tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)
    enc["labels"] = batch["label"]
    return enc

tokenized = dataset.map(
    tokenize, batched=True,
    remove_columns=["text", "pmid", "label"]
)
print("Tokenization complete.")

Map: 100%|██████████| 48848/48848 [00:06<00:00, 7941.97 examples/s]

Tokenization complete.


In [6]:
# ==========================================
# MODEL + LoRA
# ==========================================
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=["query", "key", "value"],
    lora_dropout=LORA_DROPOUT, bias="none",
    task_type=TaskType.SEQ_CLS,
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at C:\Users\UMZ\.cache\huggingface\hub\models--microsoft--BiomedNLP-PubMedBERT-base-uncased-abstract\snapshots\d673b8835373c6fa116d6d8006b33d48734e305d and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 443,906 || all params: 108,681,220 || trainable%: 0.4084


In [7]:
# ==========================================
# TRAIN
# ==========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    return {"accuracy": accuracy_score(labels, preds),
            "precision": p, "recall": r, "f1": f}

training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print(f"✅ Model saved to {MODEL_DIR}")

c:\Users\UMZ\anaconda3\envs\pubmedbert\lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
c:\Users\UMZ\anaconda3\envs\pubmedbert\lib\site-packages\accelerate\accelerator.py:477: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
  0%|          | 50/79326 [00:04<1:58:08, 11.18it/s]

{'loss': 0.426, 'grad_norm': 1.872081995010376, 'learning_rate': 0.0001998739379270353, 'epoch': 0.0}


  0%|          | 102/79326 [00:09<2:04:36, 10.60it/s]

{'loss': 0.4259, 'grad_norm': 3.79396390914917, 'learning_rate': 0.00019974787585407054, 'epoch': 0.0}


  0%|          | 152/79326 [00:14<2:04:52, 10.57it/s]

...

100%|█████████▉| 79301/79326 [2:07:34<00:02, 10.58it/s]

{'loss': 0.1799, 'grad_norm': 9.376737594604492, 'learning_rate': 1.4875324609837885e-07, 'epoch': 3.0}


                                                       
100%|██████████| 79326/79326 [2:09:20<00:00, 11.40it/s]c:\Users\UMZ\anaconda3\envs\pubmedbert\lib\site-packages\peft\utils\save_and_load.py:195: UserWarning: Could not find a config file in C:\Users\UMZ\.cache\huggingface\hub\models--microsoft--BiomedNLP-PubMedBERT-base-uncased-abstract\snapshots\d673b8835373c6fa116d6d8006b33d48734e305d - will assume that the vocabulary was not modified.
  warnings.warn(
100%|██████████| 79326/79326 [2:09:20<00:00, 10.22it/s]

{'eval_loss': 0.22740837931632996, 'eval_accuracy': 0.9137938093678349, 'eval_precision': 0.7145480073442056, 'eval_recall': 0.8084066471163245, 'eval_f1': 0.7585851057730895, 'eval_runtime': 104.1488, 'eval_samples_per_second': 469.021, 'eval_steps_per_second': 58.628, 'epoch': 3.0}
{'train_runtime': 7760.6208, 'train_samples_per_second': 163.547, 'train_steps_per_second': 10.222, 'train_loss': 0.1979016378926542, 'epoch': 3.0}
✅ Model saved to D:\student1402\negar\final_research\phase2_model_pipeline_aware_abstrcat_clean



c:\Users\UMZ\anaconda3\envs\pubmedbert\lib\site-packages\peft\utils\save_and_load.py:195: UserWarning: Could not find a config file in C:\Users\UMZ\.cache\huggingface\hub\models--microsoft--BiomedNLP-PubMedBERT-base-uncased-abstract\snapshots\d673b8835373c6fa116d6d8006b33d48734e305d - will assume that the vocabulary was not modified.
  warnings.warn(
